# Day 16 · Advanced Python — Modules, Packages, Environments, Memory & Concurrency

**Duration:** 50–60 Minutes  ·  **Type:** Theory Class (very little code)

### Learning Outcomes
- Organise code into **modules** and **packages**, and understand how `import` finds them.
- Use **pip** and **virtual environments** to isolate project dependencies.
- Understand Python's **memory management**: references, reference counting, garbage collection.
- Distinguish **concurrency vs parallelism**, and **CPU-bound vs I/O-bound** work.
- Know what the **GIL** is and why it matters.
- Choose correctly between **threading**, **multiprocessing**, and **asyncio**.

## 1. What is a Module?

A **module** is simply **one `.py` file** containing Python code — functions, classes, variables.
Writing everything in a single file becomes unreadable, so we split code into modules and
**import** what we need.

```text
project/
├── main.py          <- uses the code
└── calculator.py    <- a MODULE (one file)
```

**Why modules?**

| Benefit | Meaning |
|---|---|
| **Reusability** | Write once, import in many programs |
| **Organisation** | Related code lives together |
| **Namespacing** | `math.pi` and `mymod.pi` never clash |
| **Maintainability** | Fix a bug in one file, everyone gets it |

**Three kinds of modules:**
1. **Built-in** — come with Python (`math`, `os`, `sys`, `random`, `json`, `datetime`).
2. **Third-party** — installed with pip (`requests`, `numpy`, `pandas`).
3. **User-defined** — the `.py` files you write yourself.

## 2. Ways to Import

```python
import math                    # whole module      -> math.sqrt(16)
import math as m               # with an alias     -> m.sqrt(16)
from math import sqrt          # one name only     -> sqrt(16)
from math import sqrt, pi      # several names
from math import sqrt as root  # name + alias      -> root(16)
from math import *             # EVERYTHING  (avoid!)
```

**Key Notes:**
- `import math` keeps the namespace — you always see *where* a name came from.
- `from math import *` pollutes your namespace and can silently **overwrite** your own variables.
  It is considered bad practice.
- A module is **executed only once** per program, the first time it is imported. Later imports
  reuse the cached copy from `sys.modules`.

In [1]:
import math as m
from math import sqrt, pi

print(m.sqrt(16), sqrt(16))    # same function, two ways of naming it
print(round(pi, 4))

4.0 4.0
3.1416


## 3. How Python Finds a Module — `sys.path`

When you write `import mymodule`, Python searches **in order**:

1. The **built-in** modules compiled into the interpreter.
2. The **current directory** (the folder of the script being run).
3. The directories in the **`PYTHONPATH`** environment variable.
4. The **site-packages** folder (where pip installs things).

All of these are visible as the list **`sys.path`**. If the module is in none of them, you get
`ModuleNotFoundError`.

> **Classic beginner trap:** naming your own file `random.py` or `math.py`. Your file is found
> *first* and shadows the real standard-library module.

In [2]:
import sys

print("first 3 search paths:")
for p in sys.path[:3]:
    print(" ", p if p else "<current directory>")

print("\nmodules already loaded:", len(sys.modules))

first 3 search paths:
  C:\Users\satya3479\AppData\Local\Programs\Python\Python314\python314.zip
  C:\Users\satya3479\AppData\Local\Programs\Python\Python314\DLLs
  C:\Users\satya3479\AppData\Local\Programs\Python\Python314\Lib

modules already loaded: 981


## 4. `__name__` and `if __name__ == "__main__"`

Every module has a built-in variable **`__name__`**:

| Situation | Value of `__name__` |
|---|---|
| File is **run directly** (`python file.py`) | `"__main__"` |
| File is **imported** by another file | the module's own name, e.g. `"calculator"` |

So the famous guard means: *"run this part only when this file is the program being executed,
not when someone imports it."*

```python
# calculator.py
def add(a, b):
    return a + b

if __name__ == "__main__":     # test code / demo
    print(add(2, 3))           # runs only on: python calculator.py
```

This lets one file be **both** a reusable library **and** a runnable script.

## 5. Useful Standard-Library Modules

Python ships with a huge standard library — *"batteries included"*.

| Module | Used for |
|---|---|
| `math` | sqrt, factorial, pi, trigonometry |
| `random` | random numbers, shuffle, choice |
| `datetime` | dates, times, differences |
| `os` | files, folders, environment variables |
| `sys` | interpreter info, arguments, `sys.path` |
| `json` | read/write JSON data |
| `re` | regular expressions |
| `collections` | `Counter`, `defaultdict`, `deque` |
| `itertools` | efficient looping tools |
| `time` | timing and delays |

Two helpers when exploring any module: **`dir(module)`** lists its names, **`help(module)`**
prints its documentation.

In [3]:
import random
from collections import Counter

random.seed(42)                                  # seed -> reproducible output
print(random.randint(1, 100))
print(Counter("mississippi").most_common(2))

82
[('i', 4), ('s', 4)]


## 6. Packages — Folders of Modules

A **package** is a **folder** containing modules, used to group related code.
Historically it needed an **`__init__.py`** file to mark it as a package.

```text
mypackage/
├── __init__.py        <- marks the folder as a package (may be empty)
├── math_utils.py
├── string_utils.py
└── data/              <- sub-package
    ├── __init__.py
    └── loader.py
```

**Importing from a package**

```python
import mypackage.math_utils
from mypackage import math_utils
from mypackage.math_utils import add
from mypackage.data.loader import load_csv     # sub-package
```

**Key Notes:**
- `__init__.py` runs when the package is first imported — a good place for package-level setup
  or for re-exporting names.
- **Module = one file. Package = a folder of modules.**
- A **library / framework** is just a larger collection of packages (e.g. `numpy`, `django`).

## 7. Absolute vs Relative Imports

| Type | Example | Notes |
|---|---|---|
| **Absolute** | `from mypackage.data.loader import load_csv` | Full path from the project root — clear and **preferred** |
| **Relative** | `from .loader import load_csv` | Relative to the current module: `.` = same package, `..` = parent |

Relative imports are shorter but only work **inside a package** — running such a file directly
raises `ImportError: attempted relative import with no known parent package`.
Prefer absolute imports unless you have a reason not to.

## 8. pip and PyPI

**PyPI** (Python Package Index) is the public repository of third-party packages.
**pip** is the tool that installs from it.

| Command | Meaning |
|---|---|
| `pip install requests` | Install the latest version |
| `pip install requests==2.31.0` | Install an exact version |
| `pip install --upgrade requests` | Upgrade |
| `pip uninstall requests` | Remove |
| `pip list` | Show installed packages |
| `pip show requests` | Details of one package |
| `pip freeze > requirements.txt` | Save the exact environment |
| `pip install -r requirements.txt` | Recreate that environment |

**`requirements.txt`** is how a project records its dependencies so teammates (and servers) can
reproduce the same setup.

## 9. Virtual Environments — the *Why*

Installing everything globally creates **dependency conflicts**:

```text
Project A needs Django 3.2
Project B needs Django 5.0      -> only ONE can be installed globally
```

A **virtual environment** is an isolated folder with its **own** Python interpreter and its
**own** `site-packages`. Each project gets its own sandbox.

```text
Global Python
├── venv (Project A) -> Django 3.2
└── venv (Project B) -> Django 5.0      no conflict
```

**Benefits:** isolation · reproducibility · a clean `pip freeze` · no need for admin rights ·
easy deletion (just delete the folder).

## 10. Creating and Using a Virtual Environment

```bash
# 1. create  (creates a folder named venv)
python -m venv venv

# 2. activate
venv\Scripts\activate        # Windows
source venv/bin/activate      # macOS / Linux

# prompt becomes:  (venv) C:\project>

# 3. install packages INSIDE the environment
pip install requests

# 4. record dependencies
pip freeze > requirements.txt

# 5. leave the environment
deactivate
```

**Key Notes:**
- **Never commit the `venv/` folder to Git** — add it to `.gitignore`. Commit
  `requirements.txt` instead.
- Anything installed while the venv is active affects **only** that project.
- `sys.prefix` tells you which environment you are currently running in.

In [4]:
import sys

print("Python  :", sys.version.split()[0])
print("Env path:", sys.prefix)          # differs when a venv is active

Python  : 3.14.0rc3
Env path: C:\Users\satya3479\AppData\Local\Programs\Python\Python314


## 11. Memory Management — the Big Picture

In Python **you never allocate or free memory yourself**. The interpreter does it for you.

**Everything is an object**, and a variable is only a **name pointing to** an object:

```text
    x = 10

    x ──────► [ int object: 10 ]
    y = x                 ▲
    y ────────────────────┘        two names, ONE object
```

Memory is split into two areas:

| Area | Holds |
|---|---|
| **Stack** | function calls and the *references* (names) |
| **Heap** | the actual objects |

Python manages the heap through: **reference counting** + a **cyclic garbage collector**,
on top of its own **private heap** and **memory pools** (`pymalloc`) for small objects.

## 12. Reference Counting

Every object keeps a counter of **how many names point to it**.

- Count **increases**: assignment, adding to a list, passing to a function.
- Count **decreases**: `del name`, reassignment, a name going out of scope.
- When the count reaches **0**, the object is destroyed **immediately**.

`sys.getrefcount(obj)` shows the count — it always reads **one higher** than expected because
the argument passed to the function is itself a temporary reference.

In [5]:
import sys

a = ["data"]                       # 1 reference: a
print("after a      :", sys.getrefcount(a))

b = a                              # 2 references: a, b  (same object!)
print("after b = a  :", sys.getrefcount(a))
print("same object? :", a is b, "| id:", id(a) == id(b))

del b                              # back to 1
print("after del b  :", sys.getrefcount(a))

after a      : 2
after b = a  : 3
same object? : True | id: True
after del b  : 2


## 13. Garbage Collection and Reference Cycles

Reference counting alone cannot free a **cycle** — objects that point at each other keep each
other's count above zero even when nobody else can reach them:

```text
    a ──► [obj A] ──► [obj B]
                ◄────────┘        unreachable, but count never hits 0
```

For this, Python has a **generational garbage collector** (the `gc` module) that periodically
finds and frees unreachable cycles.

**Generational** means objects are grouped into 3 generations. New objects (gen 0) are checked
often; objects that survive are promoted and checked less often — based on the observation that
*most objects die young*.

| Function | Purpose |
|---|---|
| `gc.collect()` | Force a collection now |
| `gc.get_count()` | Objects tracked per generation |
| `gc.disable()` | Turn the cycle collector off (rare, for tuning) |

In [6]:
import gc

class Node:
    def __init__(self):
        self.other = None

gc.collect()                       # clean slate first

x = Node(); y = Node()
x.other = y; y.other = x           # reference CYCLE
del x, y                           # unreachable, but refcount != 0

print("objects freed by gc:", gc.collect())

objects freed by gc: 2


## 14. Things That Affect Memory

**Interning / caching** — small integers (`-5` to `256`) and short strings are pre-created and
**shared**, so `is` may surprise you. Compare values with `==`, identity with `is`.

**Mutable vs immutable** — modifying a list happens *in place* (same object); "modifying" a
string or tuple creates a **new** object.

**Common memory leaks in Python** (memory that is never released):
- Global lists/dicts that only ever grow (caches without a limit).
- Objects kept alive by long-lived references you forgot about.
- Circular references holding `__del__` methods or C-extension resources.

**Useful tools:** `sys.getsizeof(obj)` (size in bytes), `id(obj)` (identity),
`tracemalloc` (track allocations), generators instead of lists for big data.

In [7]:
import sys

a, b = 100, int("100")             # small ints (-5..256) are cached and SHARED
print("small int  :", a is b)

x, y = 1000, int("1000")           # large ints -> separate objects
print("large int  :", x is y, "| equal:", x == y)

nums_list = [i for i in range(10_000)]
nums_gen  = (i for i in range(10_000))     # lazy -> tiny
print("list bytes :", sys.getsizeof(nums_list))
print("gen  bytes :", sys.getsizeof(nums_gen))

small int  :

 True
large int  : False | equal: True
list bytes : 85176
gen  bytes : 200


## 15. Process vs Thread

| | **Process** | **Thread** |
|---|---|---|
| What | A running program | A unit of execution *inside* a process |
| Memory | Its **own** memory space | **Shares** the process's memory |
| Cost | Heavy to create | Light to create |
| Crash | Isolated — one crash doesn't kill others | Can bring down the whole process |
| Communication | Needs IPC (pipes, queues) | Direct (shared variables) |

**Concurrency vs Parallelism**

- **Concurrency** = *dealing with* many tasks at once by switching between them (one cook
  juggling several dishes).
- **Parallelism** = *doing* many tasks at the same instant on multiple CPU cores (several cooks).

## 16. The GIL (Global Interpreter Lock)

CPython has a **single lock** that allows **only one thread to execute Python bytecode at a
time**, even on a multi-core CPU.

**Consequences:**

| Workload | Meaning | Do threads help? |
|---|---|---|
| **I/O-bound** | Waiting on network, disk, database, `input()` | ✅ **Yes** — the GIL is *released* while waiting |
| **CPU-bound** | Heavy computation, loops, number crunching | ❌ **No** — use **multiprocessing** instead |

So: **threads for waiting, processes for computing.**

> Note: the GIL is an implementation detail of *CPython*. Recent versions ship an experimental
> free-threaded build, and other implementations (Jython, PyPy variants) differ.

## 17. Threading — Small Example

The `threading` module runs functions concurrently in the same process.

| Function | Purpose |
|---|---|
| `Thread(target=f, args=(...))` | Create a thread |
| `t.start()` | Begin running it |
| `t.join()` | Wait for it to finish |
| `Lock()` | Protect shared data from race conditions |
| `threading.current_thread().name` | Which thread am I? |

Because threads share memory, two threads updating the same variable can **corrupt** it — a
**race condition**. A **`Lock`** makes the update atomic.

In [8]:
import threading, time

def download(name, seconds):
    time.sleep(seconds)                 # simulates WAITING (I/O)
    print(f"{name} finished")

start = time.perf_counter()
threads = [threading.Thread(target=download, args=(f"file{i}", d))
           for i, d in enumerate([0.1, 0.2, 0.3])]

for t in threads: t.start()             # all three wait at the same time
for t in threads: t.join()              # wait for all to finish

print(f"total: {time.perf_counter() - start:.1f}s   (0.6s if done one by one)")

file0 finished


file1 finished
file2 finished
total: 0.3s   (0.6s if done one by one)


In [9]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100_000):
        with lock:                      # only one thread inside at a time
            counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)
t1.start(); t2.start(); t1.join(); t2.join()

print("counter:", counter, "(without the lock this is often wrong)")

counter: 200000 (without the lock this is often wrong)


## 18. Multiprocessing — True Parallelism

`multiprocessing` starts **separate Python processes**, each with its own interpreter and its
own GIL — so CPU-bound work really runs **in parallel** on multiple cores.

```python
from multiprocessing import Process, Pool

def square(n):
    return n * n

if __name__ == "__main__":            # REQUIRED on Windows
    with Pool(4) as pool:             # 4 worker processes
        print(pool.map(square, [1, 2, 3, 4, 5]))
```

| Tool | Purpose |
|---|---|
| `Process(target=f)` | One extra process |
| `Pool(n)` | A pool of `n` workers, `.map()` splits the work |
| `Queue`, `Pipe` | Send data between processes |
| `Value`, `Array` | Genuinely shared memory |
| `cpu_count()` | Number of available cores |

**Key Notes:**
- Processes **do not share memory** — data is **pickled** and copied, which costs time.
  Only worth it when the computation is heavier than the copying.
- The `if __name__ == "__main__":` guard is **mandatory** on Windows/macOS (spawn start method),
  otherwise child processes re-import the file and spawn infinitely.
- This is why the code above is shown but **not executed** here — worker processes cannot import
  a notebook cell as a module.

In [10]:
import multiprocessing as mp

print("CPU cores available:", mp.cpu_count())

CPU cores available: 12


## 19. Async / `asyncio` — Concurrency in ONE Thread

**Asynchronous programming** uses a single thread and an **event loop**. Whenever a task has to
*wait*, it hands control back to the loop, which runs another task meanwhile — **cooperative
multitasking**.

```text
sync :  |--task1--||--task2--||--task3--|      wait, wait, wait
async:  |--task1----|
        |--task2----|                          all waiting together
        |--task3----|
```

| Keyword / Function | Meaning |
|---|---|
| `async def` | Defines a **coroutine** (calling it returns a coroutine object, it doesn't run) |
| `await` | Pause here, let others run, resume when the result is ready |
| `asyncio.run(main())` | Start the event loop (from a normal `.py` script) |
| `asyncio.gather(*tasks)` | Run many coroutines concurrently and collect results |
| `asyncio.sleep(n)` | Non-blocking sleep (`time.sleep` would block everything!) |

**Key Notes:**
- `await` only works **inside** an `async def` function.
- Never call a **blocking** function (`time.sleep`, heavy loops, plain `requests.get`) inside a
  coroutine — it freezes the whole event loop.
- Async shines for **many concurrent I/O operations**: web servers, API calls, scrapers.

In [11]:
import asyncio, time

async def fetch(name, seconds):
    await asyncio.sleep(seconds)        # non-blocking wait
    return f"{name} done"

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(     # run all three concurrently
        fetch("api1", 0.3),
        fetch("api2", 0.3),
        fetch("api3", 0.3),
    )
    print(results)
    print(f"total: {time.perf_counter() - start:.1f}s   (0.9s if done one by one)")

# In a .py file you would write:  asyncio.run(main())
# Jupyter already runs an event loop, so we simply await it here:
await main()

['api1 done', 'api2 done', 'api3 done']
total: 0.3s   (0.9s if done one by one)


## 20. Which One Should I Use?

| Situation | Use | Why |
|---|---|---|
| Downloading 100 files / API calls | **asyncio** (or threads) | Mostly waiting on I/O |
| Reading many files, DB queries | **threading** | GIL is released during I/O |
| Image processing, ML training, big loops | **multiprocessing** | Needs real CPU parallelism |
| A simple script | **plain sequential code** | Concurrency adds complexity — don't pay for it |

```text
Is the task waiting (I/O)?  ── yes ──► asyncio / threading
             │
             no (CPU work)  ─────────► multiprocessing
```

**High-level shortcut:** `concurrent.futures` gives one API for both —
`ThreadPoolExecutor` (I/O) and `ProcessPoolExecutor` (CPU), both with `.map()` / `.submit()`.

## 21. Common Mistakes

**1. Naming your file after a standard module** (`random.py`, `json.py`) — your file shadows the
real one and imports break confusingly.

**2. `from module import *`** — hides where names come from and can overwrite your variables.

**3. Installing packages globally instead of in a venv** — leads to version conflicts between
projects.

**4. Committing the `venv/` folder to Git** — huge and machine-specific. Commit
`requirements.txt`.

**5. Using threads for CPU-bound work** — the GIL means it can be *slower* than a single thread.

**6. Forgetting `if __name__ == "__main__":` with multiprocessing** — infinite process spawning
on Windows.

**7. Calling `time.sleep()` inside a coroutine** — blocks the whole event loop; use
`await asyncio.sleep()`.

**8. Forgetting `await`** — calling an `async def` function without `await` just creates a
coroutine object; the body never runs.

**9. Assuming `del x` frees memory** — it only removes **one reference**; the object dies when
the count reaches 0.

## 22. Summary

- A **module** is one `.py` file; a **package** is a folder of modules (with `__init__.py`).
- `import` searches **`sys.path`**; `__name__ == "__main__"` separates script code from library code.
- **pip** installs from **PyPI**; **virtual environments** isolate each project's dependencies,
  and `requirements.txt` makes them reproducible.
- Python manages memory with **reference counting** plus a **generational garbage collector**
  for reference cycles. Variables are **names pointing to objects**.
- **Concurrency** = switching between tasks; **parallelism** = doing them at the same instant.
- The **GIL** allows only one thread to run Python bytecode at a time → threads help **I/O-bound**
  work, not **CPU-bound** work.
- **threading** → I/O in one process · **multiprocessing** → real CPU parallelism ·
  **asyncio** → thousands of I/O tasks in one thread.

## 23. Practice Questions

1. Create a module `calculator.py` with `add`, `sub`, `mul`, `div` and import it from another file.
2. Show three different ways to import `sqrt` from `math`.
3. Explain in your own words what `sys.path` is and why file naming matters.
4. Write a module that prints something only when run directly, not when imported.
5. Create a package `utils/` with two modules and import a function from each.
6. What is the difference between a module, a package, and a library?
7. Write the commands to create a venv, activate it, install `requests`, and freeze requirements.
8. Why must `venv/` be in `.gitignore` but `requirements.txt` must not be?
9. Use `sys.getrefcount()` to show how the count changes when you alias and delete a variable.
10. Create a reference cycle and free it with `gc.collect()`.
11. Explain the GIL and give one task where threads help and one where they do not.
12. Run three `time.sleep(1)` functions with threads and measure the total time.
13. Fix a race condition on a shared counter using `threading.Lock`.
14. Use `multiprocessing.Pool` to square a list of numbers in parallel (in a `.py` file).
15. Rewrite question 12 using `asyncio.gather` and `asyncio.sleep`.